### Padding to 5s and sampling rate set to 22050

In [1]:
# =======================================================
# STEP 1: Setup and Imports
# =======================================================
import os
import librosa
import numpy as np
import soundfile as sf
import pandas as pd
from IPython.display import Audio, display

# =======================================================
# STEP 2: Dataset Path
# =======================================================
# If your dataset is in Drive, mount it
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/Main Project/audio_dataset"   # 🔹 change to your dataset folder
OUTPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/Main Project/audio_dataset_padded"  # 🔹 new folder for padded audios
os.makedirs(OUTPUT_PATH, exist_ok=True)

# =======================================================
# STEP 3: Function to inspect and pad
# =======================================================
def inspect_and_pad(dataset_path, output_path, target_sr=22050, target_duration=5.0):
    file_info = []
    target_length = int(target_sr * target_duration)

    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith('.wav') or file.endswith('.mp3'):
                file_path = os.path.join(root, file)
                rel_folder = os.path.basename(root)

                # Load audio (original sample rate)
                y, sr = librosa.load(file_path, sr=None)
                duration = librosa.get_duration(y=y, sr=sr)

                # Resample if needed
                if sr != target_sr:
                    y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
                    sr = target_sr

                # Pad only if shorter than target
                if len(y) < target_length:
                    pad_length = target_length - len(y)
                    y = np.pad(y, (0, pad_length), mode='constant')

                # Save padded version (same folder structure)
                output_folder = os.path.join(output_path, rel_folder)
                os.makedirs(output_folder, exist_ok=True)
                sf.write(os.path.join(output_folder, file), y, sr)

                file_info.append({
                    "File": file,
                    "Folder": rel_folder,
                    "Original Duration (sec)": round(duration, 3),
                    "Sample Rate": sr,
                    "Padded": len(y) < target_length
                })

    df = pd.DataFrame(file_info)
    return df

# =======================================================
# STEP 4: Run and Inspect
# =======================================================
df_audio = inspect_and_pad(DATASET_PATH, OUTPUT_PATH)
df_audio.head(20)

# =======================================================
# STEP 5: Summary
# =======================================================
print("\n--- Dataset Summary ---")
print("Total files:", len(df_audio))
print("Unique sample rates:", df_audio['Sample Rate'].unique())
print("Average duration before padding (sec):", round(df_audio['Original Duration (sec)'].mean(), 2))

# =======================================================
# STEP 6: Play one padded audio
# =======================================================
sample_file = df_audio.iloc[0]['File']
sample_folder = df_audio.iloc[0]['Folder']
sample_path = os.path.join(OUTPUT_PATH, sample_folder, sample_file)
print(f"\nPlaying padded sample: {sample_file}")
display(Audio(sample_path))



--- Dataset Summary ---
Total files: 520
Unique sample rates: [22050]
Average duration before padding (sec): 4.51

Playing padded sample: 1-88807-A-39.wav
